In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import optuna
import tensorflow as tf

import warnings

In [19]:
df = pd.read_csv('../data/CORCCACBS.csv')


a = df.groupby(df['date'])['value'].mean().round(2)#.reset_index()
b = pd.to_datetime(df['date'].unique())
df= pd.DataFrame({
    'date':b,
    'value':a.values
})
df = df.set_index('date')


def traintestsplit(df):
    train = df.loc[:'2015']
    val = df.loc['2016':'2019']
    test = df.loc['2020':]

    return train,val ,test

train_set, val_set, test_set = traintestsplit(df)

Creating our timeseries data set via Tensorflow

GradDescent requires IID, shuffling training windows (not the contents)
- so the model's weight doesn't update / get biased toward whatever pattern is most recent!

In [30]:
seq_length = 56

train_dataset = tf.keras.utils.timeseries_dataset_from_array(
    train_set,targets = train_set[seq_length:],#targets are 3 steps into the future
    sequence_length=seq_length, batch_size=4,shuffle = True, seed=11
    )

val_dataset = tf.keras.utils.timeseries_dataset_from_array(
    val_set, targets=val_set[seq_length:], sequence_length=seq_length, batch_size=4, shuffle= True, seed=11
)

test_dataset = tf.keras.utils.timeseries_dataset_from_array(
    test_set, targets=test_set[seq_length:], sequence_length=seq_length, batch_size=4,shuffle= True, seed=11
)

Trying a basic linear model first: Using Huber Loss

In [32]:
tf.random.set_seed(11)

model = tf.keras.Sequential([
    tf.keras.layers.Dense(1,input_shape=[seq_length])
])

early_stopping_cb = tf.keras.callbacks.EarlyStopping(monitor='val_mae',
                                                     patience=50,
                                                     restore_best_weights=True)

opt = tf.keras.optimizers.SGD(learning_rate=.02,momentum=0.9)


model.compile(loss=tf.keras.losses.Huber(),optimizer=opt,metrics=['mae'])


history = model.fit(train_dataset, validation_data=val_dataset, epochs=500,
                    callbacks=[early_stopping_cb])

Epoch 1/500
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 9.7556 - mae: 10.2527 
Epoch 2/500
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 16.3224 - mae: 16.8223


c:\Users\Marwa\anaconda3\envs\cc_abs\Lib\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()
c:\Users\Marwa\anaconda3\envs\cc_abs\Lib\site-packages\keras\src\callbacks\early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_mae` which is not available. Available metrics are: loss,mae
  current = self.get_monitor_value(logs)


Epoch 3/500
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 32.3739 - mae: 32.8639
Epoch 4/500
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 30.0879 - mae: 30.5755 
Epoch 5/500
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 22.7724 - mae: 23.2724 
Epoch 6/500
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 41.3450 - mae: 41.8357 
Epoch 7/500
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 40.7585 - mae: 41.2585 
Epoch 8/500
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 29.3485 - mae: 29.8461 
Epoch 9/500
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 29.6715 - mae: 30.1715 
Epoch 10/500
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 11.1816 - mae: 11.6730
Epoch 11/500
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 21.9891 - mae: 22.4882
Epoch 12/500
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 30.3493 - mae: 30.8492 
Epoch 13/500
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 29.7368 - mae: 30.2368 
Epoch 14/500
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 31.5152 - mae: 32.0152
Epo